In [21]:
BATCH_SIZE = 16
LABEL= "Friendly"

In [ ]:
%load_ext autoreload
%autoreload 2
import os
from hireverse.utils.dataset_handler import DatasetHandler
from hireverse.utils.utils import BASE_DIR

train_ids = DatasetHandler.get_participant_ids()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import gc
import cv2
import numpy as np


def participant_frames_generator(participant_ids, frames_number=16):
    for participant_id in participant_ids:
        label = DatasetHandler.get_labels_dict(participant_id)[LABEL]
        number_frames = DatasetHandler.get_number_of_frames(participant_id)
        frame_yielder =  DatasetHandler.yield_sorted_participant_frames_images(participant_id, is_image_greyscale=True)
        for j in range(0, number_frames, frames_number):
            frames_batch = []
            for i in range(frames_number):
                try:
                    frame = next(frame_yielder)
                    frame = frame.astype('float32') / 255.0  # normalize
                    frames_batch.append(frame)
                except StopIteration:
                    break
            if frames_batch:
                frames_batch, label

def to_100class(score_1_to_10):
    return np.floor((score_1_to_10 - 1) * 10 + np.random.uniform(0, 10))

participants_gens = {participant_id: DatasetHandler.yield_sorted_participant_frames_images(participant_id, is_image_greyscale=True) for participant_id in train_ids}
def train_generator(train_ids, batch_size=6):
    for participant_id in train_ids:
        number_frames = DatasetHandler.get_number_of_frames(participant_id)
        participant_gen = participants_gens[participant_id]
        for j in range(0, number_frames, batch_size):
            participant_video_batch = []
            for i in range(batch_size):
                try:
                    frame_label_tuple = next(participant_gen)
                    participant_video_batch.append(frame_label_tuple)
                except StopIteration:
                    break
            if participant_video_batch:
                yield participant_video_batch

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: use group split
train_ids, temp_ids = train_test_split(train_ids, test_size=0.5, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

X, y = 


In [41]:
X, y = next(train_gen)
print(X.shape)
print(y.shape)

(16, 640, 640)
()


In [25]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_competency_cnn(input_shape=(640, 640, 1), num_classes=600):
    model = models.Sequential()

    # Initial Convolutional Blocks
    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.25))

    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.25))

    # Final Convolutional Layers
    model.add(layers.Conv2D(2048, (3, 3), activation='relu'))
    model.add(layers.GlobalAveragePooling2D())

    # Final Classification Layer
    model.add(layers.Dense(num_classes, activation='softmax'))  # 6 competencies × 100 classes

    # Compile with learning rate (as per paper)
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model

# Initialize model
model = build_competency_cnn()
model.summary()

/Users/bassel27/personal_projects/hireverse/venv/lib/python3.9/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 638, 638, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 319, 319, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 319, 319, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 319, 319, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 317, 317, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 158, 158, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 158, 158, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 158, 158, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 156, 156, 2048) │     1,181,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 600)            │     1,229,400 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,430,296 (9.27 MB)

 Trainable params: 2,430,104 (9.27 MB)

 Non-trainable params: 192 (768.00 B)